Chunking + dataset class + dataloader  

SEQ_LEN = 32      # Maximum input context  
TARGET = variable # Remaining tokens until <EOS>  
STRIDE = 16

In [1]:
import pickle

In [2]:
with open('/content/encoded_train.pkl','rb') as f:
  encoded_train=pickle.load(f)

with open('/content/encoded_test.pkl','rb') as f:
  encoded_test=pickle.load(f)

with open('/content/encoded_valid.pkl','rb') as f:
  encoded_valid=pickle.load(f)

with open('/content/vocab.pkl','rb') as f:
  vocab=pickle.load(f)

In [3]:
import torch
from torch.utils.data import Dataset

class DatasetPrep(Dataset):

    def __init__(self, data, vocab,
                 seq_length=32,
                 max_output_length=50,
                 stride=16):

        self.data = data
        self.vocab = vocab
        self.seq_length = seq_length
        self.max_output_length = max_output_length

        # Each element = (row_number, split_position)
        self.indices = []

        for row_idx, row in enumerate(data):
            for i in range(seq_length, len(row), stride):
                self.indices.append((row_idx, i))

    def prep_data(self, row, i):

        # ---------- Encoder ----------
        input_seq = row[i-self.seq_length:i]

        encoder_input = (
            [self.vocab["<BOS>"]]
            + input_seq
            + [self.vocab["<EOS>"]]
        )

        # ---------- Find newline ----------
        nearest_line_end = len(row)

        for j in range(i, len(row)):
            if row[j] == self.vocab["<NEWLINE>"]:
                nearest_line_end = j
                break

        end = min(i + self.max_output_length,
                  nearest_line_end + 1)

        labels = row[i:end]

        # ---------- Decoder ----------
        decoder_input = [self.vocab["<BOS>"]] + labels
        decoder_output = labels + [self.vocab["<EOS>"]]

        return (
            torch.tensor(encoder_input),
            torch.tensor(decoder_input),
            torch.tensor(decoder_output)
        )

    def __getitem__(self, idx):

        row_idx, i = self.indices[idx]

        row = self.data[row_idx]

        return self.prep_data(row, i)

    def __len__(self):
        return len(self.indices)

In [4]:
from torch.nn.utils.rnn import pad_sequence

PAD = vocab["<PAD>"]

def collate_fn(batch):

    encoder_inputs, decoder_inputs, decoder_outputs = zip(*batch)

    encoder_inputs = pad_sequence(
        encoder_inputs,
        batch_first=True,
        padding_value=PAD
    )

    decoder_inputs = pad_sequence(
        decoder_inputs,
        batch_first=True,
        padding_value=PAD
    )

    decoder_outputs = pad_sequence(
        decoder_outputs,
        batch_first=True,
        padding_value=PAD
    )

    return encoder_inputs, decoder_inputs, decoder_outputs

In [5]:
from torch.utils.data import DataLoader

BATCH_SIZE = 64

train_dataset = DatasetPrep(encoded_train, vocab)
valid_dataset = DatasetPrep(encoded_valid, vocab)
test_dataset = DatasetPrep(encoded_test, vocab)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    collate_fn=collate_fn
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=64,
    shuffle=False,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    collate_fn=collate_fn
)

In [6]:
encoder, decoder_in, decoder_out = next(iter(train_loader))

print(encoder.shape)
print(decoder_in.shape)
print(decoder_out.shape)

torch.Size([64, 34])
torch.Size([64, 25])
torch.Size([64, 25])


output → hidden state at every time step (batch_size, seq_len, hidden_dim)  
hidden → final hidden state for each layer (num_layers, batch_size, hidden_dim)  
cell → final cell state for each layer (num_layers, batch_size, hidden_dim)  

In [7]:
from torch import nn

class LSTMEncoder(nn.Module):
  def __init__(self, vocab_size, hidden_dim, n_layers, dropout):
    super().__init__()
    self.encoder_embedding = nn.Embedding(vocab_size, hidden_dim)
    self.encoder=nn.LSTM(hidden_dim, hidden_dim, n_layers, dropout=dropout, batch_first=True)

  def forward(self,src):
    emb = self.encoder_embedding(src)
    output, (hidden, cell) = self.encoder(emb)

    return hidden, cell

In [8]:
class LSTMDecoder(nn.Module):
  def __init__(self, vocab_size, hidden_dim, n_layers, dropout):
    super().__init__()
    self.decoder_embedding = nn.Embedding(vocab_size, hidden_dim)
    self.decoder=nn.LSTM(hidden_dim, hidden_dim, n_layers, dropout=dropout if n_layers>1 else 0, batch_first=True)
    self.linear = nn.Linear(hidden_dim, vocab_size)

  def forward(self, input_token, hidden, cell):
    input_token = input_token.unsqueeze(1)
    emb = self.decoder_embedding(input_token)
    output, (h_c, c_c) = self.decoder(emb,(hidden,cell))
    pred = self.linear(output.squeeze(1))

    return pred, h_c, c_c

In [9]:
class seq2seq(nn.Module):
  def __init__(self, encoder, decoder):
    super().__init__()
    self.encoder = encoder
    self.decoder = decoder

  def forward(self, src, target, teacher_forcing_ratio = 0.5):
    batch_size = src.shape[0]
    target_len = target.shape[1]
    target_vocab_size = self.decoder.linear.out_features

    outputs = torch.zeros(batch_size, target_len, target_vocab_size, device=src.device)

    hidden, cell = self.encoder(src)

    input_token = target[:,0] #<SOS>

    for t in range(1, target_len):
      pred, hidden, cell = self.decoder(input_token, hidden, cell)
      outputs[:,t,:] = pred

      teacher_force = torch.rand(1).item() < teacher_forcing_ratio

      top1 = pred.argmax(dim=1)

      input_token = target[:, t] if teacher_force else top1

    return outputs

Loss function - crossEntropyLoss

In [10]:
criterion = nn.CrossEntropyLoss(ignore_index=vocab['<PAD>'])

In [11]:
def token_accuracy(pred, target):
  # ignore <SOS> token
  pred = pred[:,1:,:]
  target = target[:,1:]

  # (B,S,V)->(B,S) for pred to compare pred and target
  pred = pred.argmax(dim=2)

  # mask to ignore indices with padding
  mask = target != vocab['<PAD>']

  # calculate correct tokens
  correct = (pred == target) & mask

  return correct.sum().float() / mask.sum().float()

In [12]:
epochs=20
vocab_size=len(vocab)
hidden_dim=256
n_layers=2
dropout=0.3
linear=vocab_size
clip=1.0
PAD_IDX=vocab['<PAD>']
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [13]:
encoder = LSTMEncoder(vocab_size, hidden_dim, n_layers, dropout)
decoder = LSTMDecoder(vocab_size, hidden_dim, n_layers, dropout)
model = seq2seq(encoder, decoder)
model=model.to(device)

In [14]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [15]:
device

device(type='cuda')

In [16]:
from torch.utils.data import Subset

small_dataset = Subset(train_dataset, range(5000))
small_loader = DataLoader(
    small_dataset,
    batch_size=64,
    shuffle=True,
    collate_fn=collate_fn
)

In [17]:
!pip install optuna -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 26.3 MB/s eta 0:00:00


In [18]:
import optuna
import gc

In [19]:
from torch.utils.data import DataLoader, Subset

# Small training subset already used in your notebook
small_train_dataset = Subset(
    train_dataset,
    range(min(5000, len(train_dataset)))
)

# Small validation subset
small_valid_dataset = Subset(
    valid_dataset,
    range(min(1000, len(valid_dataset)))
)

small_train_loader = DataLoader(
    small_train_dataset,
    batch_size=64,
    shuffle=True,
    collate_fn=collate_fn
)

small_valid_loader = DataLoader(
    small_valid_dataset,
    batch_size=64,
    shuffle=False,
    collate_fn=collate_fn
)

In [20]:
import math


def evaluate_model(model, loader, criterion, device):

    model.eval()

    total_loss = 0.0

    # Token-level metrics
    total_correct = 0
    total_tokens = 0

    # Top-5 token accuracy
    total_top5_correct = 0

    # Exact match
    total_sequences = 0
    total_exact_matches = 0

    with torch.no_grad():

        for encoder_inputs, decoder_inputs, decoder_outputs in loader:

            encoder_inputs = encoder_inputs.to(device)
            decoder_inputs = decoder_inputs.to(device)
            decoder_outputs = decoder_outputs.to(device)

            outputs = model(
                encoder_inputs,
                decoder_inputs,
                teacher_forcing_ratio=0.0
            )

            # =====================================
            # Loss
            # =====================================

            pred = outputs[:, 1:, :].reshape(
                -1,
                outputs.shape[-1]
            )

            target = decoder_outputs[:, 1:].reshape(-1)

            loss = criterion(pred, target)

            total_loss += loss.item()

            # =====================================
            # Predictions
            # =====================================

            predictions = outputs[:, 1:, :].argmax(dim=2)

            target_tokens = decoder_outputs[:, 1:]

            # Don't count PAD tokens
            mask = target_tokens != PAD_IDX

            # =====================================
            # Token Accuracy
            # =====================================

            total_correct += (
                (predictions == target_tokens) & mask
            ).sum().item()

            total_tokens += mask.sum().item()

            # =====================================
            # Top-5 Accuracy
            # =====================================

            top5_predictions = outputs[:, 1:, :].topk(
                k=5,
                dim=2
            ).indices

            target_expanded = target_tokens.unsqueeze(-1)

            top5_correct = (
                (top5_predictions == target_expanded).any(dim=-1)
                & mask
            )

            total_top5_correct += top5_correct.sum().item()

            # =====================================
            # Exact Match Accuracy
            # =====================================

            sequence_correct = (
                ((predictions == target_tokens) | ~mask)
                .all(dim=1)
            )

            total_exact_matches += (
                sequence_correct.sum().item()
            )

            total_sequences += decoder_outputs.size(0)

    # =========================================
    # Final Metrics
    # =========================================

    avg_loss = total_loss / len(loader)

    token_accuracy = (
        total_correct / total_tokens
        if total_tokens > 0
        else 0.0
    )

    top5_accuracy = (
        total_top5_correct / total_tokens
        if total_tokens > 0
        else 0.0
    )

    exact_match_accuracy = (
        total_exact_matches / total_sequences
        if total_sequences > 0
        else 0.0
    )

    # Perplexity
    perplexity = math.exp(avg_loss)

    return (
        avg_loss,
        token_accuracy,
        top5_accuracy,
        exact_match_accuracy,
        perplexity
    )

In [21]:
def objective(trial):

    # ==========================
    # 1. Hyperparameters
    # ==========================

    hidden_dim = trial.suggest_categorical(
        "hidden_dim",
        [128, 256, 384, 512]
    )

    n_layers = trial.suggest_int(
        "n_layers",
        1,
        3
    )

    dropout = trial.suggest_float(
        "dropout",
        0.1,
        0.5
    )

    learning_rate = trial.suggest_float(
        "learning_rate",
        1e-4,
        3e-3,
        log=True
    )

    weight_decay = trial.suggest_float(
        "weight_decay",
        1e-6,
        1e-3,
        log=True
    )

    teacher_forcing_ratio = trial.suggest_float(
        "teacher_forcing_ratio",
        0.3,
        0.7
    )

    # ==========================
    # 2. Build model
    # ==========================

    encoder = LSTMEncoder(
        vocab_size,
        hidden_dim,
        n_layers,
        dropout
    )

    decoder = LSTMDecoder(
        vocab_size,
        hidden_dim,
        n_layers,
        dropout
    )

    model = seq2seq(
        encoder,
        decoder
    ).to(device)

    # ==========================
    # 3. Loss
    # ==========================

    criterion = nn.CrossEntropyLoss(
        ignore_index=vocab["<PAD>"]
    )

    # ==========================
    # 4. Optimizer
    # ==========================

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )

    # ==========================
    # 5. Scheduler
    # ==========================

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=1
    )

    # ==========================
    # 6. Train trial
    # ==========================

    NUM_EPOCHS = 5

    best_val_loss = float("inf")

    for epoch in range(NUM_EPOCHS):

        model.train()

        for encoder_inputs, decoder_inputs, decoder_outputs in small_train_loader:

            encoder_inputs = encoder_inputs.to(device)
            decoder_inputs = decoder_inputs.to(device)
            decoder_outputs = decoder_outputs.to(device)

            optimizer.zero_grad()

            outputs = model(
                encoder_inputs,
                decoder_inputs,
                teacher_forcing_ratio
            )

            # Ignore timestep 0 because the model
            # does not generate a prediction there
            pred = outputs[:, 1:, :].reshape(
                -1,
                outputs.shape[-1]
            )

            target = decoder_outputs[:, 1:].reshape(-1)

            loss = criterion(pred, target)

            loss.backward()

            # Prevent exploding gradients
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimizer.step()

        # ==========================
        # Validation
        # ==========================

        (
            val_loss,
            val_token_accuracy,
            val_top5_accuracy,
            val_exact_match,
            val_perplexity
        ) = evaluate_model(
            model,
            small_valid_loader,
            criterion,
            device
        )

        scheduler.step(val_loss)

        best_val_loss = min(
            best_val_loss,
            val_loss
        )

        # ==========================
        # Optuna pruning
        # ==========================

        trial.report(
            val_loss,
            epoch
        )

        if trial.should_prune():

            del model
            gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            raise optuna.TrialPruned()

    # Cleanup GPU memory
    del model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return best_val_loss

In [22]:
study = optuna.create_study(
    study_name="lstm_code_completion",
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.HyperbandPruner()
)

[I 2026-10-07 17:37:51,029] A new study created in memory with name: lstm_code_completion


In [23]:
study.optimize(
    objective,
    n_trials=10
)

/usr/local/lib/python3.13/dist-packages/torch/nn/modules/rnn.py:1013: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.16239780813448107 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)
[I 2026-10-07 17:39:35,636] Trial 0 finished with value: 4.340545862913132 and parameters: {'hidden_dim': 256, 'n_layers': 1, 'dropout': 0.16239780813448107, 'learning_rate': 0.00012184186502221769, 'weight_decay': 0.0003967605077052988, 'teacher_forcing_ratio': 0.5404460046972834}. Best is trial 0 with value: 4.340545862913132.
/usr/local/lib/python3.13/dist-packages/torch/nn/modules/rnn.py:1013: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.17272998688284025 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)
[I 2026-10-07 17:41:41,529] Trial 1 finished with value: 4.340378150343

In [24]:
print("Best validation loss:")
print(study.best_value)

print("\nBest hyperparameters:")

for key, value in study.best_params.items():
    print(f"{key}: {value}")

Best validation loss:
4.335348516702652

Best hyperparameters:
hidden_dim: 384
n_layers: 1
dropout: 0.29807076404450805
learning_rate: 0.00011240768803005555
weight_decay: 0.0005345166110646819
teacher_forcing_ratio: 0.40351199264000676


In [25]:
df = study.trials_dataframe()

df[
    [
        "number",
        "value",
        "params_hidden_dim",
        "params_n_layers",
        "params_dropout",
        "params_learning_rate",
        "params_weight_decay",
        "params_teacher_forcing_ratio",
        "state"
    ]
]

,number,value,params_hidden_dim,params_n_layers,params_dropout,params_learning_rate,params_weight_decay,params_teacher_forcing_ratio,state
0,0,4.340546,256,1,0.162398,0.000122,0.000397,0.540446,COMPLETE
1,1,4.340378,384,1,0.172730,0.000187,0.000008,0.509903,COMPLETE
2,2,4.401560,384,1,0.246545,0.000472,0.000227,0.379870,PRUNED
3,3,4.569605,512,1,0.126021,0.002521,0.000789,0.623359,PRUNED
4,4,4.335349,384,1,0.298071,0.000112,0.000535,0.403512,COMPLETE
5,5,4.396513,128,1,0.487834,0.001396,0.000658,0.657931,COMPLETE
6,6,4.358205,256,1,0.230132,0.000375,0.000007,0.631495,COMPLETE
7,7,4.637416,384,3,0.129820,0.002869,0.000207,0.379486,PRUNED
8,8,4.432910,256,3,0.129618,0.000338,0.000002,0.645241,PRUNED
9,9,4.392065,128,1,0.391842,0.000874,0.000459,0.488886,PRUNED


In [26]:
best_params = study.best_params

print(best_params)

{'hidden_dim': 384, 'n_layers': 1, 'dropout': 0.29807076404450805, 'learning_rate': 0.00011240768803005555, 'weight_decay': 0.0005345166110646819, 'teacher_forcing_ratio': 0.40351199264000676}


In [ ]:
# {'hidden_dim': 384, 'n_layers': 1, 'dropout': 0.29807076404450805, 'learning_rate': 0.00011240768803005555, 'weight_decay': 0.0005345166110646819, 'teacher_forcing_ratio': 0.40351199264000676}

In [27]:
PAD_IDX = vocab["<PAD>"]

In [28]:
def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    device,
    teacher_forcing_ratio
):
    model.train()

    epoch_loss = 0.0
    total_correct = 0
    total_tokens = 0

    for encoder_inputs, decoder_inputs, decoder_outputs in loader:

        encoder_inputs = encoder_inputs.to(device)
        decoder_inputs = decoder_inputs.to(device)
        decoder_outputs = decoder_outputs.to(device)

        optimizer.zero_grad()

        outputs = model(
            encoder_inputs,
            decoder_inputs,
            teacher_forcing_ratio
        )

        # Loss
        pred = outputs[:, 1:, :].reshape(
            -1, outputs.shape[-1]
        )

        target = decoder_outputs[:, 1:].reshape(-1)

        loss = criterion(pred, target)

        # Accuracy
        predictions = outputs[:, 1:, :].argmax(dim=2)
        target_tokens = decoder_outputs[:, 1:]

        mask = target_tokens != PAD_IDX

        correct = (
            (predictions == target_tokens) & mask
        ).sum()

        total = mask.sum()

        total_correct += correct.item()
        total_tokens += total.item()

        # Backpropagation
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        epoch_loss += loss.item()

    epoch_accuracy = (
        total_correct / total_tokens
        if total_tokens > 0
        else 0.0
    )

    return (
        epoch_loss / len(loader),
        epoch_accuracy
    )

In [29]:
def validate(
    model,
    loader,
    criterion,
    device
):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_tokens = 0

    with torch.no_grad():

        for encoder_inputs, decoder_inputs, decoder_outputs in loader:

            encoder_inputs = encoder_inputs.to(device)
            decoder_inputs = decoder_inputs.to(device)
            decoder_outputs = decoder_outputs.to(device)

            outputs = model(
                encoder_inputs,
                decoder_inputs,
                teacher_forcing_ratio=0.0
            )

            # Ignore timestep 0 because your model
            # does not generate a prediction there
            pred = outputs[:, 1:, :].reshape(
                -1, outputs.shape[-1]
            )

            target = decoder_outputs[:, 1:].reshape(-1)

            loss = criterion(pred, target)

            total_loss += loss.item()

            predictions = outputs[:, 1:, :].argmax(dim=2)
            target_tokens = decoder_outputs[:, 1:]

            mask = target_tokens != PAD_IDX

            total_correct += (
                (predictions == target_tokens) & mask
            ).sum().item()

            total_tokens += mask.sum().item()

    avg_loss = total_loss / len(loader)

    accuracy = (
        total_correct / total_tokens
        if total_tokens > 0
        else 0.0
    )

    return avg_loss, accuracy

In [31]:
# =========================
# FINAL MODEL
# =========================

hidden_dim = best_params["hidden_dim"]
n_layers = best_params["n_layers"]
dropout = best_params["dropout"]

encoder = LSTMEncoder(
    vocab_size,
    hidden_dim,
    n_layers,
    dropout
)

decoder = LSTMDecoder(
    vocab_size,
    hidden_dim,
    n_layers,
    dropout
)

model = seq2seq(
    encoder,
    decoder
).to(device)

print("Final model created.")
print(f"Hidden dimension : {hidden_dim}")
print(f"LSTM layers      : {n_layers}")
print(f"Dropout          : {dropout}")

Final model created.
Hidden dimension : 384
LSTM layers      : 1
Dropout          : 0.29807076404450805


In [32]:
criterion = nn.CrossEntropyLoss(
    ignore_index=vocab["<PAD>"]
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=best_params["learning_rate"],
    weight_decay=best_params["weight_decay"]
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

In [ ]:
NUM_EPOCHS = 20
PATIENCE = 4

best_val_loss = float("inf")
epochs_without_improvement = 0

train_losses = []
val_losses = []

train_accuracies = []
val_accuracies = []

learning_rates = []

BEST_MODEL_PATH = "best_lstm_code_completion.pt"


for epoch in range(NUM_EPOCHS):

    # -------------------------
    # Training
    # -------------------------
    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        device,
        best_params["teacher_forcing_ratio"]
    )

    # -------------------------
    # Validation
    # -------------------------
    val_loss, val_acc = validate(
        model,
        valid_loader,
        criterion,
        device
    )

    # -------------------------
    # Scheduler
    # -------------------------
    scheduler.step(val_loss)

    current_lr = optimizer.param_groups[0]["lr"]

    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)

    learning_rates.append(current_lr)

    # -------------------------
    # Print
    # -------------------------
    print(
        f"Epoch [{epoch+1:02d}/{NUM_EPOCHS}] "
        f"| Train Loss: {train_loss:.4f} "
        f"| Val Loss: {val_loss:.4f} "
        f"| Train Acc: {train_acc*100:.2f}% "
        f"| Val Acc: {val_acc*100:.2f}% "
        f"| LR: {current_lr:.6f}"
    )

    # -------------------------
    # Save best model
    # -------------------------
    if val_loss < best_val_loss:

        best_val_loss = val_loss
        epochs_without_improvement = 0

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "best_val_loss": best_val_loss,
                "best_params": best_params
            },
            BEST_MODEL_PATH
        )

        print("   ✓ Best model saved")

    else:

        epochs_without_improvement += 1

        print(
            f"   No improvement "
            f"({epochs_without_improvement}/{PATIENCE})"
        )

        if epochs_without_improvement >= PATIENCE:
            print("Early stopping triggered.")
            break

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(train_losses, label="Training Loss")
plt.plot(val_losses, label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("LSTM Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(train_accuracies, label="Training Accuracy")
plt.plot(val_accuracies, label="Validation Accuracy")

plt.xlabel("Epoch")
plt.ylabel("Token Accuracy")
plt.title("LSTM Training and Validation Accuracy")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=device
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

print(
    f"Best validation loss: "
    f"{checkpoint['best_val_loss']:.4f}"
)

print("\nBest Hyperparameters:")
for key, value in checkpoint["best_params"].items():
    print(f"{key}: {value}")

In [ ]:
test_loss, test_accuracy = validate(
    model,
    test_loader,
    criterion,
    device
)

print("\n===== FINAL LSTM TEST RESULTS =====")
print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_accuracy * 100:.2f}%")